In [ ]:
!git clone https://github.com/d4-5/NLP4.git
%cd NLP4

In [ ]:
!pip install -r requirements.txt

In [ ]:
%cd NLP4

In [23]:
from pathlib import Path


from pathlib import Path
import sys

REPO_ROOT = Path('..').resolve()
DATA_DIR = REPO_ROOT / 'data'
sys.path.append(str(REPO_ROOT))
LABELS_PATH = DATA_DIR / 'labels.csv'
SPLIT_DIR = DATA_DIR / 'sample'
DOCS_DIR = REPO_ROOT / "docs"
SPLIT_MANIFEST_PATH = DOCS_DIR / "splits_manifest_lab5.json"

In [24]:
import sys

import pandas as pd
from IPython.display import Markdown, display

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from src.topic_modeling import (
    DEFAULT_TOPIC_COUNTS,
    build_topic_stop_words,
    top_documents_table,
    topic_words_table,
    run_lda_experiments,
    run_lsa_experiments,
    summarize_topic_runs,
)
from src.topic_utils import (
    collect_short_examples,
    filter_corpus,
    prepare_corpus_frame,
    top_token_counts,
)

processed_path = REPO_ROOT / "data" / "processed_v2.csv"
df = pd.read_csv(processed_path)
df.head(3)


,text_id,text_clean,sentences
0,text_0,"Дорогою зупинялися в селах, старцювали. Устимк...","['Дорогою зупинялися в селах, старцювали.', 'У..."
1,text_1,"То молитви продавав свої спудейські, то щось к...","['То молитви продавав свої спудейські, то щось..."
2,text_2,"Дуже часто дід досить точно вказував, куди сам...","['Дуже часто дід досить точно вказував, куди с..."


In [25]:
prepared_df = prepare_corpus_frame(df, text_col="text_clean")

summary_before = pd.DataFrame(
    [
        {
            "documents_before_filtering": len(prepared_df),
            "empty_or_blank": int(prepared_df["text_clean"].eq("").sum()),
            "min_token_len": int(prepared_df["token_len"].min()),
            "median_token_len": float(prepared_df["token_len"].median()),
            "mean_token_len": float(prepared_df["token_len"].mean()),
            "max_token_len": int(prepared_df["token_len"].max()),
        }
    ]
)
summary_before


,documents_before_filtering,empty_or_blank,min_token_len,median_token_len,mean_token_len,max_token_len
0,11161,0,0,13.0,16.437237,202


In [26]:
MIN_TOKENS = 5
MIN_CHARS = 20

filtered_df, filter_stats = filter_corpus(
    df,
    text_col="text_clean",
    min_tokens=MIN_TOKENS,
    min_chars=MIN_CHARS,
)

pd.DataFrame([filter_stats])


,documents_before,documents_after,dropped_total,dropped_empty,dropped_short,min_tokens,min_chars
0,11161,9930,1231,0,1231,5,20


In [27]:
removed_mask = ~prepared_df["text_id"].isin(filtered_df["text_id"])
prepared_df.loc[
    removed_mask,
    ["text_id", "text_clean", "token_len", "char_len"],
].sort_values(["token_len", "char_len"]).head(10)


,text_id,text_clean,token_len,char_len
1513,text_1513,* * *,0,5
4535,text_4535,* * *,0,5
124,text_124,З,1,1
3198,text_3198,4,1,1
3209,text_3209,5,1,1
6926,text_6926,4,1,1
6951,text_6951,5,1,1
81,text_81,1.,1,2
87,text_87,2.,1,2
93,text_93,3.,1,2


In [28]:
print("Shortest texts in the original corpus:")
display(collect_short_examples(df, text_col="text_clean", limit=10))

print("Top tokens before filtering:")
display(top_token_counts(prepared_df, text_col="text_clean", top_n=20))

print("Top tokens after filtering:")
display(top_token_counts(filtered_df, text_col="text_clean", top_n=20))


Shortest texts in the original corpus:


,text_id,text_clean,token_len,char_len
0,text_1513,* * *,0,5
1,text_4535,* * *,0,5
2,text_3198,4,1,1
3,text_6926,4,1,1
4,text_3209,5,1,1
5,text_6951,5,1,1
6,text_124,З,1,1
7,text_81,1.,1,2
8,text_2273,1.,1,2
9,text_2624,1.,1,2


Top tokens before filtering:


,token,count
0,і,3651
1,на,3505
2,у,2959
3,не,2674
4,в,2618
5,з,2345
6,що,2074
7,та,1608
8,до,1593
9,а,1296


Top tokens after filtering:


,token,count
0,і,3594
1,на,3474
2,у,2950
3,в,2604
4,не,2588
5,з,2329
6,що,2035
7,та,1595
8,до,1565
9,а,1257


In [29]:
extra_stop_words = [
    "це",
    "так",
    "же",
    "ще",
    "просто",
    "один",
    "одна",
    "одне",
    "одного",
]

topic_stop_words = build_topic_stop_words(extra_stop_words=extra_stop_words)

pd.DataFrame(
    {
        "stop_word": topic_stop_words[:25],
    }
)


,stop_word
0,email
1,http
2,https
3,phone
4,url
5,www
6,а
7,або
8,аж
9,але


In [30]:
TOPIC_COUNTS = list(DEFAULT_TOPIC_COUNTS)

lsa_vectorizer_params = {
    "analyzer": "word",
    "ngram_range": (1, 1),
    "min_df": 3,
    "max_df": 0.9,
    "token_pattern": r"(?u)\b[\w']+\b",
}

lda_vectorizer_params = {
    "analyzer": "word",
    "ngram_range": (1, 1),
    "min_df": 3,
    "max_df": 0.9,
    "token_pattern": r"(?u)\b[\w']+\b",
}

corpus_texts = filtered_df["text_clean"].tolist()

pd.DataFrame(
    [
        {
            "model": "LSA",
            **lsa_vectorizer_params,
            "stop_words_count": len(topic_stop_words),
            "topic_counts": TOPIC_COUNTS,
        },
        {
            "model": "LDA",
            **lda_vectorizer_params,
            "stop_words_count": len(topic_stop_words),
            "topic_counts": TOPIC_COUNTS,
        },
    ]
)


,model,analyzer,ngram_range,min_df,max_df,token_pattern,stop_words_count,topic_counts
0,LSA,word,"(1, 1)",3,0.9,(?u)\b[\w']+\b,129,"[5, 8]"
1,LDA,word,"(1, 1)",3,0.9,(?u)\b[\w']+\b,129,"[5, 8]"


In [31]:
lsa_runs = run_lsa_experiments(
    texts=corpus_texts,
    topic_counts=TOPIC_COUNTS,
    vectorizer_params=lsa_vectorizer_params,
    stop_words=topic_stop_words,
    random_state=42,
)
lsa_runs_by_k = {run.topic_count: run for run in lsa_runs}


In [32]:
lsa_summary = summarize_topic_runs(lsa_runs)
lsa_summary


,model,k,vectorizer,documents,vocab_size,matrix_density,explained_variance_ratio_sum
0,LSA,5,TfidfVectorizer,9930,9516,0.00095,0.016967
1,LSA,8,TfidfVectorizer,9930,9516,0.00095,0.022676


In [33]:
lda_runs = run_lda_experiments(
    texts=corpus_texts,
    topic_counts=TOPIC_COUNTS,
    vectorizer_params=lda_vectorizer_params,
    stop_words=topic_stop_words,
    random_state=42,
    lda_params={"max_iter": 10, "learning_method": "batch", "n_jobs": -1},
)
lda_runs_by_k = {run.topic_count: run for run in lda_runs}


In [34]:
lda_summary = summarize_topic_runs(lda_runs)
lda_summary


,model,k,vectorizer,documents,vocab_size,matrix_density,n_iter,training_bound
0,LDA,5,CountVectorizer,9930,9516,0.00095,10,6336.841510
1,LDA,8,CountVectorizer,9930,9516,0.00095,10,7300.299837


In [35]:
experiment_overview = pd.concat([lsa_summary, lda_summary], ignore_index=True)
experiment_overview


,model,k,vectorizer,documents,vocab_size,matrix_density,explained_variance_ratio_sum,n_iter,training_bound
0,LSA,5,TfidfVectorizer,9930,9516,0.00095,0.016967,NaN,NaN
1,LSA,8,TfidfVectorizer,9930,9516,0.00095,0.022676,NaN,NaN
2,LDA,5,CountVectorizer,9930,9516,0.00095,NaN,10.0,6336.841510
3,LDA,8,CountVectorizer,9930,9516,0.00095,NaN,10.0,7300.299837


In [36]:
topic_word_tables = {}

for model_name, runs_by_k in [("LSA", lsa_runs_by_k), ("LDA", lda_runs_by_k)]:
    for k in TOPIC_COUNTS:
        run = runs_by_k[k]
        table = topic_words_table(run, top_n=10)
        topic_word_tables[(model_name, k)] = table
        print(f"{model_name}, k={k}")
        display(table)


LSA, k=5


,model,k,topic_id,top_words,top_word_weights
0,LSA,5,0,"віснику, повідомляється, закупівель, державних...","0.5024, 0.5002, 0.4980, 0.4912, 0.0386, 0.0301..."
1,LSA,5,1,"я, є, грн, тов, тебе, млн, бо, року, цього, під","0.8490, 0.1600, 0.1227, 0.1081, 0.0988, 0.0932..."
2,LSA,5,2,"грн, тов, млн, є, року, 1, тис, фірма, вартіст...","0.4014, 0.3208, 0.3098, 0.2820, 0.1855, 0.1395..."
3,LSA,5,3,"є, навіть, власником, тов, бо, нього, хто, тіл...","0.7017, 0.1028, 0.0856, 0.0810, 0.0766, 0.0704..."
4,LSA,5,4,"є, я, тов, власником, директором, фірми, засно...","0.3471, 0.2343, 0.2163, 0.0550, 0.0498, 0.0466..."


LSA, k=8


,model,k,topic_id,top_words,top_word_weights
0,LSA,8,0,"віснику, повідомляється, закупівель, державних...","0.5024, 0.5002, 0.4980, 0.4912, 0.0386, 0.0301..."
1,LSA,8,1,"я, є, грн, тов, тебе, млн, бо, року, цього, під","0.8490, 0.1600, 0.1227, 0.1081, 0.0988, 0.0932..."
2,LSA,8,2,"грн, тов, млн, є, року, 1, тис, фірма, вартіст...","0.4014, 0.3208, 0.3098, 0.2820, 0.1855, 0.1395..."
3,LSA,8,3,"є, навіть, власником, тов, бо, нього, хто, тіл...","0.7017, 0.1028, 0.0856, 0.0811, 0.0766, 0.0705..."
4,LSA,8,4,"є, я, тов, власником, директором, фірми, засно...","0.3471, 0.2343, 0.2163, 0.0550, 0.0498, 0.0466..."
5,LSA,8,5,"суду, свідчить, рішення, року, області, господ...","0.3466, 0.3264, 0.3234, 0.2957, 0.1747, 0.1742..."
6,LSA,8,6,"тов, під, року, час, фірми, компанія, конкурен...","0.5538, 0.3057, 0.1471, 0.0983, 0.0702, 0.0568..."
7,LSA,8,7,"навіть, тов, ніколи, нього, ніхто, місто, добр...","0.6426, 0.3091, 0.1222, 0.0938, 0.0813, 0.0562..."


LDA, k=5


,model,k,topic_id,top_words,top_word_weights
0,LDA,5,0,"грн, тов, року, млн, 1, є, році, тис, області,...","529.1985, 458.3712, 433.1236, 358.1986, 211.17..."
1,LDA,5,1,"є, б, років, повідомляється, державних, нього,...","115.6111, 77.8365, 77.2423, 76.0918, 72.3208, ..."
2,LDA,5,2,"є, навіть, світ, під, час, щось, тільки, потім...","164.8892, 72.1597, 60.9491, 60.8772, 57.3659, ..."
3,LDA,5,3,"я, бо, тільки, життя, ніколи, тепер, більше, н...","566.5418, 179.1382, 140.4376, 97.3695, 91.1671..."
4,LDA,5,4,"україни, є, має, під, час, щодо, виконання, на...","94.2919, 75.0323, 59.1462, 51.4560, 41.4367, 4..."


LDA, k=8


,model,k,topic_id,top_words,top_word_weights
0,LDA,8,0,"грн, млн, року, тов, тис, 1, україни, м, рішен...","524.9653, 356.2318, 288.9127, 200.5195, 171.91..."
1,LDA,8,1,"тов, є, фірми, директором, власником, році, ро...","327.0125, 243.6916, 75.0763, 68.3914, 66.1386,..."
2,LDA,8,2,"є, під, тільки, хто, щось, ви, навіть, люди, п...","77.0138, 59.4219, 57.8936, 56.6810, 50.6765, 4..."
3,LDA,8,3,"я, бо, тільки, життя, ніколи, кілька, тебе, те...","524.0178, 143.8220, 96.8352, 88.3577, 83.6551,..."
4,LDA,8,4,"україни, ділянки, під, щодо, яку, навіть, газе...","57.2940, 51.4846, 36.2315, 35.0711, 31.9512, 3..."
5,LDA,8,5,"навіть, повідомляється, державних, очі, закупі...","87.8669, 81.1246, 76.3754, 73.9537, 68.8084, 6..."
6,LDA,8,6,"є, року, чого, роботи, зараз, управління, мину...","129.6015, 82.4470, 47.0590, 47.0081, 41.5820, ..."
7,LDA,8,7,"багато, час, буде, тепер, бо, україни, більше,...","56.0967, 53.0580, 49.5890, 48.6286, 42.7897, 4..."


In [37]:
topic_document_tables = {}

for model_name, runs_by_k in [("LSA", lsa_runs_by_k), ("LDA", lda_runs_by_k)]:
    for k in TOPIC_COUNTS:
        run = runs_by_k[k]
        table = top_documents_table(
            run,
            filtered_df,
            text_col="text_clean",
            id_col="text_id",
            top_n=2,
            preview_chars=240,
        )
        topic_document_tables[(model_name, k)] = table
        print(f"{model_name}, k={k}")
        display(table)


LSA, k=5


,model,k,topic_id,doc_rank,topic_score,text_id,text_preview
0,LSA,5,0,1,0.995874,text_10819,"Про це повідомляється у ""Віснику державних зак..."
1,LSA,5,0,2,0.995874,text_10639,"Про це повідомляється в ""Віснику державних зак..."
2,LSA,5,1,1,0.849002,text_210,"Я так, може, віддячуся Тарасові і його засніже..."
3,LSA,5,1,2,0.849002,text_4635,Я й так чувся невигідно.
4,LSA,5,2,1,0.440949,text_10640,"Найбільші підряди отримали ПАТ ""Дніпропетровсь..."
5,LSA,5,2,2,0.397262,text_10242,Тоді вартість реалізації цього проекту була 7 ...
6,LSA,5,3,1,0.701702,text_370,Є полум'яна комета з хвостом.
7,LSA,5,3,2,0.701702,text_1226,"Її нема там, де вона є."
8,LSA,5,4,1,0.347086,text_1226,"Її нема там, де вона є."
9,LSA,5,4,2,0.347086,text_370,Є полум'яна комета з хвостом.


LSA, k=8


,model,k,topic_id,doc_rank,topic_score,text_id,text_preview
0,LSA,8,0,1,0.995874,text_10819,"Про це повідомляється у ""Віснику державних зак..."
1,LSA,8,0,2,0.995874,text_10639,"Про це повідомляється в ""Віснику державних зак..."
2,LSA,8,1,1,0.849002,text_210,"Я так, може, віддячуся Тарасові і його засніже..."
3,LSA,8,1,2,0.849002,text_4635,Я й так чувся невигідно.
4,LSA,8,2,1,0.440949,text_10640,"Найбільші підряди отримали ПАТ ""Дніпропетровсь..."
5,LSA,8,2,2,0.397262,text_10242,Тоді вартість реалізації цього проекту була 7 ...
6,LSA,8,3,1,0.701700,text_370,Є полум'яна комета з хвостом.
7,LSA,8,3,2,0.701700,text_1226,"Її нема там, де вона є."
8,LSA,8,4,1,0.347096,text_1226,"Її нема там, де вона є."
9,LSA,8,4,2,0.347096,text_370,Є полум'яна комета з хвостом.


LDA, k=5


,model,k,topic_id,doc_rank,topic_score,text_id,text_preview
0,LDA,5,0,1,0.986790,text_10437,"Нагадаємо, Машненков відомий як екс-депутат До..."
1,LDA,5,0,2,0.986618,text_10606,"Це, зокрема, ""свободівці"" Андріян Гутник (підп..."
2,LDA,5,1,1,0.981830,text_6320,"Інколи здається, що кілька годин ось такого зо..."
3,LDA,5,1,2,0.978078,text_5997,"Пан Корнель чув, як обпікає його груди гаряче ..."
4,LDA,5,2,1,0.981130,text_4005,"В цьому немає нічого дивного, якщо згадати, що..."
5,LDA,5,2,2,0.980734,text_7048,Його уклав колектив співробітників Інституту у...
6,LDA,5,3,1,0.981703,text_4362,"У разі прийняття сільськими, селищними, міськи..."
7,LDA,5,3,2,0.978484,text_478,"І так нагадувала її й сама Леся, котра взяла д..."
8,LDA,5,4,1,0.984611,text_506,Другий випадок складання зведеної податкової н...
9,LDA,5,4,2,0.982836,text_9655,У рамках Загальнодержавної цільової науково-те...


LDA, k=8


,model,k,topic_id,doc_rank,topic_score,text_id,text_preview
0,LDA,8,0,1,0.985639,text_10606,"Це, зокрема, ""свободівці"" Андріян Гутник (підп..."
1,LDA,8,0,2,0.984366,text_4919,"""Зокрема у зв'язку з переміщенням в Україну ак..."
2,LDA,8,1,1,0.985160,text_9744,Відповідно до вимог Повітряного кодексу Україн...
3,LDA,8,1,2,0.979156,text_10617,"Власником і директором столичної фірми ""ІКК ""Г..."
4,LDA,8,2,1,0.982124,text_2658,Результати цих досліджень опубліковано в таких...
5,LDA,8,2,2,0.979156,text_7048,Його уклав колектив співробітників Інституту у...
6,LDA,8,3,1,0.976940,text_478,"І так нагадувала її й сама Леся, котра взяла д..."
7,LDA,8,3,2,0.975650,text_8646,"Можливо, іноді при заснуванні громад у нас із ..."
8,LDA,8,4,1,0.981368,text_9655,У рамках Загальнодержавної цільової науково-те...
9,LDA,8,4,2,0.979631,text_9762,Частина фірм обслуговує також запорізькі міськ...
